In [8]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [9]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [10]:
import json

from prompt_toolkit import prompt


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    clean_json = json.loads(text.strip())
    return clean_json

In [ ]:
# dataset = generate_dataset()
# dataset
# with open("dataset.json", "w") as f:
#     json.dump(dataset, f, indent=2)

In [12]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [13]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [14]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [15]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [16]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extractor\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_region_from_s3_url(s3_url):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Supports formats:\n    - s3://bucket-name.region.amazonaws.com/key\n    - https://bucket-name.s3.region.amazonaws.com/key\n    - https://s3.region.amazonaws.com/bucket-name/key\n    - https://bucket-name.s3-region.amazonaws.com/key\n    \n    Args:\n        s3_url (str): S3 URL in various formats\n        \n    Returns:\n        str: AWS region name, or None if region cannot be extracted\n    \"\"\"\n    if not s3_url:\n        return None\n    \n    # Remove protocol if present\n    url = s3_url.replace('s3://', '').replace('https://', '').replace('http://', '')\n    \n    # Pattern 1: bucket-name.region.amazonaws.com/key\n    match = re.search(r'\\.([a-z0-9\\-]+)\\.amazonaws\\.com', ur